# 02 - Data Preparation: Cleaning
This notebook applies the cleaning decisions identified in the data audit. I will standardize the typology labels, combine the date and time into a single timestamp, assign appropriate data types, and then save the cleaned dataset for EDA.

Connecting to Drive and saving output to my processed folder

In [1]:
from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/INFO320_AML")
RAW_FILE = PROJECT / "Data" / "Raw" / "SAML-D.csv"
CLEAN_FOLDER = PROJECT / "Data" / "Processed"

CLEAN_FOLDER.mkdir(parents=True, exist_ok=True)

print("Raw dataset found:", RAW_FILE.is_file())
print("Cleaned data folder:", CLEAN_FOLDER)

Mounted at /content/drive
Raw dataset found: True
Cleaned data folder: /content/drive/MyDrive/INFO320_AML/Data/Processed


Setting the data types for each column keeping account IDs as text and using categories for repeated labels to save memory

In [2]:
column_types = {
    "Sender_account": "string",
    "Receiver_account": "string",
    "Date": "string",
    "Time": "string",
    "Amount": "float64",
    "Is_laundering": "int8",
    "Payment_currency": "category",
    "Received_currency": "category",
    "Sender_bank_location": "category",
    "Receiver_bank_location": "category",
    "Payment_type": "category",
    "Laundering_type": "category",
}

df = pd.read_csv(RAW_FILE, dtype=column_types, keep_default_na=False)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9504852 entries, 0 to 9504851
Data columns (total 12 columns):
 #   Column                  Dtype   
---  ------                  -----   
 0   Time                    string  
 1   Date                    string  
 2   Sender_account          string  
 3   Receiver_account        string  
 4   Amount                  float64 
 5   Payment_currency        category
 6   Received_currency       category
 7   Sender_bank_location    category
 8   Receiver_bank_location  category
 9   Payment_type            category
 10  Is_laundering           int8    
 11  Laundering_type         category
dtypes: category(6), float64(1), int8(1), string(4)
memory usage: 426.0 MB


Each column now has the intended data type

Standardizing the typology labels using lowercase letters, underscores for the spaces, and correcting Foward to forward. These issues were found in the data audit.

In [3]:
typologies = set()

for chunk in pd.read_csv(
    RAW_FILE, usecols=["Laundering_type"], chunksize=250_000):
    typologies.update(chunk["Laundering_type"].unique())

label_check = pd.DataFrame({"Original": sorted(typologies)})

label_check["Cleaned"] = (
    label_check["Original"]
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .replace({"normal_foward": "normal_forward"}))

display(label_check)
print("Distinct cleaned labels:", label_check["Cleaned"].nunique())

,Original,Cleaned
0,Behavioural_Change_1,behavioural_change_1
1,Behavioural_Change_2,behavioural_change_2
2,Bipartite,bipartite
3,Cash_Withdrawal,cash_withdrawal
4,Cycle,cycle
5,Deposit-Send,deposit_send
6,Fan_In,fan_in
7,Fan_Out,fan_out
8,Gather-Scatter,gather_scatter
9,Layered_Fan_In,layered_fan_in


Distinct cleaned labels: 28


All laundering type labels are now normalized

Applying the typology changes to the df.

In [4]:
label_mapping = dict(zip(label_check["Original"], label_check["Cleaned"]))

df["Laundering_type"] = df["Laundering_type"].cat.rename_categories(label_mapping)

display(df["Laundering_type"].value_counts())
print("Distinct typologies:", df["Laundering_type"].nunique())

,count
Laundering_type,
normal_small_fan_out,3477717
normal_fan_out,2302220
normal_fan_in,2104285
normal_group,528351
normal_cash_withdrawal,305031
normal_cash_deposits,223801
normal_periodical,210526
normal_plus_mutual,155041
normal_mutual,125335


Distinct typologies: 28


Combining date and time into a single timestamp and cinverting to datetime

In [5]:

df["Timestamp"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%Y-%m-%d %H:%M:%S")

df["Date"] = df["Timestamp"].dt.normalize()

display(df[["Date", "Time", "Timestamp"]].head())
print("\nData types:")
print(df[["Date", "Timestamp"]].dtypes)


,Date,Time,Timestamp
0,2022-10-07,10:35:19,2022-10-07 10:35:19
1,2022-10-07,10:35:20,2022-10-07 10:35:20
2,2022-10-07,10:35:20,2022-10-07 10:35:20
3,2022-10-07,10:35:21,2022-10-07 10:35:21
4,2022-10-07,10:35:21,2022-10-07 10:35:21



Data types:
Date         datetime64[ns]
Timestamp    datetime64[ns]
dtype: object


Both columns now use datetime data types, allowing for accurate date filtering, chronological analysis, and to be able to have rolling transaction windows in later notebooks.

Checking all cleaned data before saving and exporting.

In [6]:
print("Transactions:", len(df))
print("Laundering transactions:", df["Is_laundering"].sum())
print("Distinct typologies:", df["Laundering_type"].nunique())
print("Missing timestamps:", df["Timestamp"].isna().sum())
print("Chronological order:", df["Timestamp"].is_monotonic_increasing)

Transactions: 9504852
Laundering transactions: 9873
Distinct typologies: 28
Missing timestamps: 0
Chronological order: True


After cleaning the dataset contains 9,504,852 transactions with 9,873 labeled as laundering and 28 typologies. Every timestamp is present and in chronological order.

Saving the cleaned dataset to be used for EDA and modeling.

In [ ]:
df.to_parquet(CLEAN_FOLDER / "SAML-D_cleaned.parquet", index=False)
df.to_csv(CLEAN_FOLDER / "SAML-D_cleaned.csv", index=False)

print("Saved cleaned Parquet and CSV files.")